In [1]:
library(SNITCH)
library(Seurat)
library(dbscan)     
library(ggplot2)
library(ggrepel)
library(dplyr)
library(tibble)
library(factoextra)

## run SNITCH on gene expression across pseudotime


In [6]:
seurat2 = readRDS('./410.DifferentialExpressions/01.ExN/111.PFC_bigger/metacells_ExN_seurat2.RDS')
meta = read.csv('./410.DifferentialExpressions/01.ExN/111.PFC_bigger/OBS_forseurat_l23_metacells2.csv')

In [16]:
genes_expr = read.csv('./400.Metacell/01.ExN//12.PFC_bigger/all_genes_expression_metacellNorm50k.csv', row.names = 1)
colnames(genes_expr) <- paste0("snitch_", colnames(genes_expr))
meta = cbind(meta, genes_expr)

Set_1_GP_factor,2.1711287,2.3424293,1.2485105,1.8571155,2.2022128,1.53628875,2.6984010,1.3024491,1.4151235,2.7653943,⋯,1.8872197,1.1195472,1.3390165,1.1662972,0.9586632,1.3214048,1.3331529,0.6852345,2.6784754,1.2136194
Set_2_GP_factor,-0.3171863,0.7539624,-0.1987924,0.3005028,-0.1951377,-0.08580618,-0.7384751,-0.4012618,-0.2688875,-1.2028934,⋯,-0.2621482,-0.6733572,-0.2540828,-0.7062438,-0.1535388,-0.4989207,-0.9657312,-0.7496694,-0.9070184,-0.6595594
Set_3_GP_factor,1.0588503,0.8522418,1.4459311,0.7416224,1.0677713,0.88496914,0.8613027,1.0768160,0.8512512,0.9590387,⋯,0.5713133,1.0027409,0.9732916,1.1180784,0.7772247,1.1019839,0.6102497,0.5170245,0.8596238,1.0635917
Set_4_GP_factor,2.3321095,1.8280572,0.9086739,1.7009563,2.0698776,1.50916937,1.8038210,1.1818698,1.4450725,1.9178056,⋯,1.3609641,0.6686943,1.6456976,0.7446095,0.8920190,1.7128831,1.3473780,0.4223234,1.6354456,1.0835042
Set_5_GP_factor,2.0310453,1.4421524,1.4141572,1.9942268,2.0441927,2.19389962,1.7211764,1.5535058,1.7848946,1.5860099,⋯,1.9664898,1.2631282,1.3685424,1.4083406,1.2198365,1.3172717,1.4408186,1.0134854,1.5166482,1.7355415
Set_6_GP_factor,2.0785674,1.2741577,2.1139698,1.5134505,1.6914163,1.90136478,1.5105452,2.3092629,2.1239440,1.5288405,⋯,1.4289610,2.2309316,2.3473705,2.3203374,1.6938854,2.5649698,1.4583459,1.7048503,1.4525093,2.4725568


In [18]:

# Select all columns starting with "mean_TOTAL_GP_"
mean_cols <- grep("GP_factor", colnames(meta), value = TRUE)

# Create a data frame with only these columns
mean_df <- meta[, mean_cols]
mean_df = t(mean_df)


In [19]:
count_df  <- mean_df
scaled_data <- prepare_data(data = t(count_df), age = seurat2@meta.data$pseudotime_rank)


In [21]:
classified_cpgs <- run_parallel_classification(dat_scaled = scaled_data$dat_scaled, 
                                               age = scaled_data$Age, 
                                               ages_grid = scaled_data$ages_grid, 
                                              covariates = meta$total_cells)

In [22]:
non_linear_cpgs <- classified_cpgs[grep("NL", classified_cpgs$classification),]$CpG
nl_smooth_rows  <- classified_cpgs[grep("NL", classified_cpgs$classification),]
nl_smooth       <- do.call(rbind, nl_smooth_rows$Predictions)
rownames(nl_smooth) <- non_linear_cpgs

# Perform FPCA
fpca_results <- perform_fpca(nl_var_smooth = nl_smooth, ages_grid = scaled_data$ages_grid)

# Plot FPCA results (files are saved by the function)
plot_fpca_results(fpca_results, ages_grid = scaled_data$ages_grid)

Warning message in sqrt(Eigen$values):
“NaNs produced”


In [25]:
class_work <- classified_cpgs
nl_mask    <- class_work$classification == "NL"


In [27]:
hdb_fit <- hdbscan(as.matrix(fpca_results$scores), minPts = 2)
pred_hdb <- paste0("NL_", hdb_fit$cluster)
cw_hdb <- class_work; cw_hdb$classification[nl_mask] <- pred_hdb
cw_hdb2 <- cw_hdb[ , !(names(cw_hdb) %in% "Predictions")]
cw_hdb2

,CpG,lm_pval,lm_coef,dbic_lg,bp_pval,white_pval,gam_edf,gam_refdf,gam_pval,adj_lm,adj_white,classification,adj_gam
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
BP,Set_1_GP_factor,1.034158e-03,3.339161e-04,36.368303566,6.534896e-02,1.828698e-01,3.284800,3.723352,0.000000e+00,1.551238e-02,1.000000e+00,NL_0,0.000000e+00
BP1,Set_2_GP_factor,5.257424e-46,-1.385215e-03,-8.764288119,6.999093e-02,3.449728e-11,3.067469,3.556076,0.000000e+00,1.104059e-44,7.244428e-10,LD,0.000000e+00
BP2,Set_3_GP_factor,1.538190e-02,2.469049e-04,-0.008390483,3.575628e-09,2.704429e-08,1.000679,1.001358,1.542242e-02,1.567098e-01,5.408858e-07,VI,6.168966e-02
BP3,Set_4_GP_factor,7.949598e-01,2.651532e-05,34.578867851,3.553575e-01,6.488812e-01,3.360121,3.774178,0.000000e+00,1.000000e+00,1.000000e+00,NL_0,0.000000e+00
BP4,Set_5_GP_factor,3.979966e-02,2.095491e-04,5.422037144,7.015280e-01,1.615988e-02,2.720071,3.237930,1.193947e-04,2.785976e-01,1.817151e-01,NL_0,1.313342e-03
BP5,Set_6_GP_factor,1.388008e-01,-1.509568e-04,-5.691176674,1.448929e-03,3.361584e-03,2.391674,2.895820,4.486353e-02,7.284599e-01,5.042375e-02,NC,1.345906e-01
BP6,Set_7_GP_factor,4.979600e-14,7.582539e-04,-6.409111651,1.302429e-02,9.816498e-06,2.788807,3.305018,0.000000e+00,9.461240e-13,1.865135e-04,LI,0.000000e+00
BP7,Set_8_GP_factor,1.821576e-01,-1.360679e-04,-6.478549983,2.887698e-01,2.798276e-01,1.991082,2.446418,1.800101e-01,7.286306e-01,1.000000e+00,NC,3.600201e-01
BP8,Set_9_GP_factor,3.251982e-12,-7.026351e-04,35.817530416,1.569035e-01,2.273093e-01,3.405801,3.802978,0.000000e+00,5.853568e-11,1.000000e+00,NL_0,0.000000e+00


In [1]:
# try to find optimal number of clusters 

p_elbow <- fviz_nbclust(fpca_results$scores, kmeans, k.max = 6, method = "wss") +
  labs(title = "Elbow Method for K-Means",
       x = "Number of Clusters (K)",
       y = "Total Within-Cluster Sum of Squares (WCSS)") +
  theme_minimal()
ggplot2::ggsave("SNITCH_kmeans_elbow_2.pdf", p_elbow, width = 6, height = 4)

k_opt  <- 5
km_fit <- kmeans(as.matrix(fpca_results$scores), centers = k_opt, nstart = 2)
pred_km <- paste0("NL_", km_fit$cluster)
cw_km <- class_work; cw_km$classification[nl_mask] <- pred_km
# df_comp <- dplyr::bind_rows(df_comp, add_metrics(cw_km$classification, "SNITCH + K-Means"))

ERROR: Error in fviz_nbclust(fpca_results$scores, kmeans, k.max = 6, method = "wss"): could not find function "fviz_nbclust"


In [33]:
cw_km2 <- cw_km[ , !(names(cw_km) %in% "Predictions")]
cw_km2_sorted<- cw_km2[order(cw_km2$adj_gam),]
subset(cw_km2_sorted, classification != "NC")

,CpG,lm_pval,lm_coef,dbic_lg,bp_pval,white_pval,gam_edf,gam_refdf,gam_pval,adj_lm,adj_white,classification,adj_gam
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
BP,Set_1_GP_factor,1.034158e-03,3.339161e-04,36.368303566,6.534896e-02,1.828698e-01,3.284800,3.723352,0.000000e+00,1.551238e-02,1.000000e+00,NL_1,0.000000e+00
BP1,Set_2_GP_factor,5.257424e-46,-1.385215e-03,-8.764288119,6.999093e-02,3.449728e-11,3.067469,3.556076,0.000000e+00,1.104059e-44,7.244428e-10,LD,0.000000e+00
BP3,Set_4_GP_factor,7.949598e-01,2.651532e-05,34.578867851,3.553575e-01,6.488812e-01,3.360121,3.774178,0.000000e+00,1.000000e+00,1.000000e+00,NL_1,0.000000e+00
BP6,Set_7_GP_factor,4.979600e-14,7.582539e-04,-6.409111651,1.302429e-02,9.816498e-06,2.788807,3.305018,0.000000e+00,9.461240e-13,1.865135e-04,LI,0.000000e+00
BP8,Set_9_GP_factor,3.251982e-12,-7.026351e-04,35.817530416,1.569035e-01,2.273093e-01,3.405801,3.802978,0.000000e+00,5.853568e-11,1.000000e+00,NL_3,0.000000e+00
BP10,Set_11_GP_factor,2.695239e-02,2.254534e-04,40.968957893,2.042033e-01,1.514293e-02,3.361002,3.774748,0.000000e+00,2.156191e-01,1.817151e-01,NL_1,0.000000e+00
BP12,Set_13_GP_factor,1.424634e-02,2.497223e-04,26.171027761,8.519423e-03,2.240862e-02,3.649657,3.927301,0.000000e+00,1.567098e-01,2.236278e-01,NL_1,0.000000e+00
BP18,Set_19_GP_factor,5.282988e-37,1.248248e-03,-5.112669574,7.182344e-01,1.365672e-02,2.613544,3.130556,0.000000e+00,1.056598e-35,1.775374e-01,LI,0.000000e+00
BP20,Set_21_GP_factor,2.118281e-10,6.419636e-04,-5.625979541,8.649667e-02,1.423582e-01,2.449978,2.958759,0.000000e+00,3.601078e-09,9.965076e-01,LI,0.000000e+00


In [100]:
write.csv(cw_km2, 'classification_modules_SNITCH.csv')